# $\Lambda_b^0 \to p K^- \gamma$

We construct an amplitude model for the radiative decay studied by LHCb {cite}`LHCb:2024vtc`, using {class}`~ampform_dpd.DalitzPlotDecompositionBuilder`. The notebook makes the pK resonance content, LS couplings, dynamics, and numerical parameters explicit. It follows {doc}`lc2pkpi` and uses the [pinned reference model](https://github.com/RUB-EP1/amplitude-serialization/blob/4bf857c9592a558943c32e42782c5f6cb90224b1/models/lb2pkg-lhcb-2765817.json) as the source of the values below.

The reference contains 13 $\Lambda^*$ resonances and a nonresonant $J^P=3/2^-$ contribution, with 74 production LS components in total. We retain their masses, widths, radii, and complex weights. The final plot illustrates this amplitude construction without detector acceptance, backgrounds, or the normalization of the experimental likelihood.

<!-- cspell:ignore isfinite LNR nonresonant -->

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import sympy as sp
from IPython.display import Markdown, Math
from matplotlib_inline.backend_inline import set_matplotlib_formats
from tensorwaves.data.transform import SympyDataTransformer

from ampform_dpd import DalitzPlotDecompositionBuilder, DefinedExpression
from ampform_dpd.decay import (
    IsobarNode,
    Particle,
    State,
    ThreeBodyDecay,
    ThreeBodyDecayChain,
)
from ampform_dpd.dynamics.builder import get_mandelstam_s
from ampform_dpd.io import as_markdown_table, aslatex, cached, simplify_latex_rendering

set_matplotlib_formats("svg")
simplify_latex_rendering()
from ampform.dynamics.form_factor import FormFactor, SphericalHankel1
from ampform.dynamics.phasespace import PhaseSpaceFactorComplex
from ampform.kinematics.phasespace import BreakupMomentumSquared
from ampform.sympy import PoolSum
from attrs import evolve

from ampform_dpd.dynamics import BreitWigner

## Decay definition

Assign indices $1=p$, $2=K^-$, and $3=\gamma$, so all resonance dynamics depend on $\sigma_3=m^2(pK^-)$. The parent has spin $1/2$ and the photon has spin one and zero mass. Masses and widths are in GeV.

The resonance table lists $(J,P,m,\Gamma,l)$, where $l$ is the orbital angular momentum in $\Lambda^*\to pK^-$. For $\Lambda(1405)$, the zero width entry is unused because its two-channel denominator is defined below. The nonresonant entry has no pole. The reference labels the $\Lambda(1670)$ propagator `L1680_BW`; its mass parameter is $1.674\,\mathrm{GeV}$, which we retain.

In [ ]:
states = {
    0: State("Lb", R"\Lambda_b^0", "1/2", 1, 5.62, 0, index=0),
    1: State("p", "p", "1/2", 1, 0.938, 0, index=1),
    2: State("K", "K^-", 0, -1, 0.493, 0, index=2),
    3: State("gamma", R"\gamma", 1, -1, 0, 0, index=3),
}
resonance_parameters = {
    "L1405": ("1/2", -1, 1.405, 0, 0),
    "L1520": ("3/2", -1, 1.519, 0.016, 2),
    "L1600": ("1/2", 1, 1.6, 0.2, 1),
    "L1670": ("1/2", -1, 1.674, 0.03, 0),
    "L1690": ("3/2", -1, 1.69, 0.07, 2),
    "L1800": ("1/2", -1, 1.8, 0.2, 0),
    "L1810": ("1/2", 1, 1.79, 0.11, 1),
    "L1820": ("5/2", 1, 1.82, 0.08, 3),
    "L1830": ("5/2", -1, 1.825, 0.09, 2),
    "L1890": ("3/2", 1, 1.89, 0.12, 1),
    "L2100": ("7/2", -1, 2.1, 0.2, 4),
    "L2110": ("5/2", 1, 2.09, 0.25, 3),
    "L2350": ("9/2", 1, 2.35, 0.15, 5),
    "LNR3O": ("3/2", -1, 0, 0, 2),
}
resonances = {
    name: Particle(
        name,
        R"\mathrm{NR}_{3/2^-}" if name == "LNR3O" else Rf"\Lambda({name[1:]})",
        spin,
        parity,
        mass,
        width,
    )
    for name, (spin, parity, mass, width, _) in resonance_parameters.items()
}
Markdown(as_markdown_table(list(resonances.values())))

### Production LS components

At the weak radiative vertex, several $(L,S)$ combinations contribute to each resonance. At the strong vertex, the proton and kaon have total spin $1/2$ and the orbital angular momentum $l$ is fixed by the resonance spin and parity.

The following weights multiply the product of the production and decay LS recoupling factors. Each key is a resonance name and its production $(L,S)$; the decay indices are always $(l,1/2)$. They are copied from the reference model in its LS convention. This is why the vertex-factor normalization below must match that convention.

These 74 components are not independent fit parameters: the publication relates the highest-$S$ couplings to the others to account for the photon's missing longitudinal state. We preserve the resulting reference weights and explicitly sum only over the two transverse photon helicities below.

In [ ]:
ls_weights = {
    ("L1405", 0, "1/2"): 2.2615419999999995 - 1.7992479999999997j,
    ("L1405", 1, "1/2"): -1.8715784466594079 - 1.0439686414131668j,
    ("L1405", 1, "3/2"): 1.3234058111554519 + 0.7381973056893572j,
    ("L1405", 2, "3/2"): -1.6131550895688382 + 1.283401355622205j,
    ("L1520", 0, "1/2"): 4.108357043622831 - 0.0668483592242114j,
    ("L1520", 1, "1/2"): 5.529490890260219 + 0.8175133786512758j,
    ("L1520", 1, "3/2"): -1.669745451910645 + 21.202140697865232j,
    ("L1520", 2, "3/2"): -5.803165988479272 + 10.325839240068499j,
    ("L1520", 2, "5/2"): 5.0240140735133085 - 3.4922186723465236j,
    ("L1520", 3, "5/2"): 4.749762166036306 - 6.557080951190571j,
    ("L1600", 0, "1/2"): -4.984604534495612 - 7.275774343593705j,
    ("L1600", 1, "1/2"): 5.195985372142845 - 0.4417798231845672j,
    ("L1600", 1, "3/2"): -3.6741164915883107 + 0.3123855087652013j,
    ("L1600", 2, "3/2"): 3.5567875127680906 + 5.1916622777038715j,
    ("L1670", 0, "1/2"): -0.16420699999999996 + 0.07874799999999998j,
    ("L1670", 1, "1/2"): 0.3371436476130202 + 0.2063535379997733j,
    ("L1670", 1, "3/2"): -0.23839655946113422 - 0.14591398604147554j,
    ("L1670", 2, "3/2"): 0.1171887985742462 - 0.05619969617692755j,
    ("L1690", 0, "1/2"): -1.2515408516146889 - 0.20807601551425958j,
    ("L1690", 1, "1/2"): 6.390591891759433 + 5.314838910034677j,
    ("L1690", 1, "3/2"): 4.392938701500898 + 1.1832477556172771j,
    ("L1690", 2, "3/2"): 8.02152561159109 + 6.075105479215116j,
    ("L1690", 2, "5/2"): -3.615370170843966 - 2.1815697682335045j,
    ("L1690", 3, "5/2"): 3.351576416899477 + 3.6239307230788382j,
    ("L1800", 0, "1/2"): 0.9999999999999999 + 0.0j,
    ("L1800", 1, "1/2"): 0.32174415383329497 - 4.420345983044671j,
    ("L1800", 1, "3/2"): -0.22750747298265048 + 3.1256566198016014j,
    ("L1800", 2, "3/2"): -0.7138719815230252 + 0.0j,
    ("L1810", 0, "1/2"): 1.2329512295731921 + 0.5797348113863545j,
    ("L1810", 1, "1/2"): -0.15041712217563452 + 0.31809866486267147j,
    ("L1810", 1, "3/2"): 0.10636096709695654 - 0.22492972301078187j,
    ("L1810", 2, "3/2"): -0.8801479601319633 - 0.4138463869619736j,
    ("L1820", 1, "3/2"): 7.590027925605289 + 1.351485280203232j,
    ("L1820", 2, "3/2"): 12.6978984716254 + 33.13909642765735j,
    ("L1820", 2, "5/2"): 4.086834167925889 + 24.975212258662644j,
    ("L1820", 3, "5/2"): -9.182639071211478 + 32.96932660214078j,
    ("L1820", 3, "7/2"): 8.508305899110448 - 6.222783113391748j,
    ("L1820", 4, "7/2"): 9.94392149899061 + 22.674983152315246j,
    ("L1830", 1, "3/2"): -1.0036485528418424 + 1.0430833903661603j,
    ("L1830", 2, "3/2"): -1.8458695010259276 - 0.7346836306540477j,
    ("L1830", 2, "5/2"): 1.6142828743499704 + 1.2089369146899427j,
    ("L1830", 3, "5/2"): -1.454215075121507 - 0.9561291141558163j,
    ("L1830", 3, "7/2"): -0.5284079103486713 + 1.100915689156118j,
    ("L1830", 4, "7/2"): -1.9512990733056674 - 0.9063591893327532j,
    ("L1890", 0, "1/2"): -0.8056899642094871 + 1.594995260820461j,
    ("L1890", 1, "1/2"): -1.1187921784182446 - 0.19731651443448214j,
    ("L1890", 1, "3/2"): 1.254387267694874 - 0.02961475092130131j,
    ("L1890", 2, "3/2"): -0.44068377083734417 - 3.386298247036518j,
    ("L1890", 2, "5/2"): -0.4595157065063295 + 2.3292545983400124j,
    ("L1890", 3, "5/2"): -1.2730779559824494 - 0.13950620539243158j,
    ("L2100", 2, "5/2"): 49.218173237760695 - 29.272299895852452j,
    ("L2100", 3, "5/2"): 51.79177222101752 + 112.49297256009743j,
    ("L2100", 3, "7/2"): 14.467132844964842 + 76.34331465657102j,
    ("L2100", 4, "7/2"): -59.296784189872376 + 160.73881774317323j,
    ("L2100", 4, "9/2"): 54.38718320352659 - 53.55523743422298j,
    ("L2100", 5, "9/2"): 44.59238847485171 + 88.989649321906j,
    ("L2110", 1, "3/2"): 7.0777507748244615 - 6.718928308295796j,
    ("L2110", 2, "3/2"): 20.666121772608257 + 20.610990545268972j,
    ("L2110", 2, "5/2"): 1.0221059150313465 + 10.97293097970227j,
    ("L2110", 3, "5/2"): -0.8039914494040608 + 32.728514478798054j,
    ("L2110", 3, "7/2"): 6.207189601628342 - 13.040156799647997j,
    ("L2110", 4, "7/2"): 17.508583975005255 + 15.177934637395214j,
    ("L2350", 3, "7/2"): 8.633355975060676 + 32.74590935567872j,
    ("L2350", 4, "7/2"): -26.4986029240209 + 36.770502872109034j,
    ("L2350", 4, "9/2"): -26.681900472924337 + 115.3793814748836j,
    ("L2350", 5, "9/2"): -56.25554547348509 + 16.55083422280756j,
    ("L2350", 5, "11/2"): 15.75763657303104 + 28.47907714973581j,
    ("L2350", 6, "11/2"): -21.36666536366439 + 18.419929714546964j,
    ("LNR3O", 0, "1/2"): 0.2014761778704208 + 0.1373786839543982j,
    ("LNR3O", 1, "1/2"): -0.912800424869022 + 0.009663096700249658j,
    ("LNR3O", 1, "3/2"): 0.7796215552309834 - 0.004829994231255104j,
    ("LNR3O", 2, "3/2"): -0.3239760701193479 - 0.11244089269832912j,
    ("LNR3O", 2, "5/2"): 0.1587446676919606 + 0.07208653021070943j,
    ("LNR3O", 3, "5/2"): -0.31776619398253025 + 0.0029782945639327254j,
}

In [ ]:
chains = [
    ThreeBodyDecayChain(
        IsobarNode(
            states[0],
            IsobarNode(
                resonances[name],
                states[1],
                states[2],
                interaction=(resonance_parameters[name][4], "1/2"),
            ),
            states[3],
            interaction=(orbital, spin),
        )
    )
    for name, orbital, spin in ls_weights
]
decay = ThreeBodyDecay(states, chains)
assert len(decay.chains) == 74
Math(aslatex(decay.find_chain("L1520"), with_jp=True))

## Lineshapes for dynamics

The resonances other than $\Lambda(1405)$ use the unity-numerator running-width Breit–Wigner,
$$
\mathcal R(s)=\frac{1}{m_R^2-s-i m_R\Gamma_R(s)}.
$$

Both vertices carry Blatt–Weisskopf factors evaluated at the running pK mass, with $R_{\Lambda_b}=5$ and $R_\mathrm{res}=1.5$ in $\mathrm{GeV}^{-1}$. Their normalization is the unnormalized convention of the reference model, obtained by dividing AmpForm's form factor by $|h_L^{(1)}(1)|$. The running width uses the ratio of the decay factors at $s$ and at the pole, so this constant cancels there.

For $\Lambda(1405)$, use
$$
\mathcal R_{1405}(s)=
\frac{1}{m_{1405}^2-s-i g^2\left[\rho_{pK}(s)+\rho_{\Sigma\pi}(s)\right]},
\qquad g^2=0.24941478752959237\,\mathrm{GeV}^2,
$$
with $m_\Sigma=1.197$ and $m_\pi=0.14$ GeV. The phase-space factors are analytically continued below threshold.

The nonresonant term has a constant propagator and momentum factors $q_\mathrm{prod}^{L}q_\mathrm{dec}^{2}$ instead of Blatt–Weisskopf factors.

The builder returns each dynamics function as a named subexpression. This keeps the angular amplitudes readable and lets the numerical transformer evaluate each distinct dynamics function once per grid point.

In [ ]:
def formulate_dynamics(chain: ThreeBodyDecayChain) -> DefinedExpression:
    s = get_mandelstam_s(chain.decay_node)
    m_parent, m_proton, m_kaon = sp.symbols("m0 m1 m2", nonnegative=True)
    r_parent, r_decay = sp.symbols("R_Lb R_res", nonnegative=True)
    mass = sp.Symbol(f"m_{{{chain.resonance.latex}}}", nonnegative=True)
    width = sp.Symbol(Rf"\Gamma_{{{chain.resonance.latex}}}", nonnegative=True)
    assert chain.incoming_ls is not None
    assert chain.outgoing_ls is not None
    orbital = chain.incoming_ls.L
    decay_orbital = chain.outgoing_ls.L
    parameters = {r_parent: 5.0, r_decay: 1.5}
    if chain.resonance.name == "LNR3O":
        production = BreakupMomentumSquared(m_parent**2, sp.sqrt(s), 0)
        decay_momentum = BreakupMomentumSquared(s, m_proton, m_kaon)
        expression = production ** sp.Rational(orbital, 2) * decay_momentum
    else:
        parameters[mass] = chain.resonance.mass
        if chain.resonance.name == "L1405":
            coupling_squared, m_sigma, m_pion = sp.symbols(
                "g_1405^2 m_Sigma m_pi", nonnegative=True
            )
            parameters.update({
                coupling_squared: 0.24941478752959237,
                m_sigma: 1.197,
                m_pion: 0.14,
            })
            channel_sum = coupling_squared * (
                PhaseSpaceFactorComplex(s, m_proton, m_kaon)
                + PhaseSpaceFactorComplex(s, m_sigma, m_pion)
            )
            propagator = 1 / (mass**2 - s - sp.I * channel_sum)
        else:
            parameters[width] = chain.resonance.width
            propagator = BreitWigner(
                s, mass, width, m_proton, m_kaon, decay_orbital, r_decay
            )
        production = unnormalized_form_factor(
            m_parent**2, sp.sqrt(s), 0, orbital, r_parent
        )
        decay_factor = unnormalized_form_factor(
            s, m_proton, m_kaon, decay_orbital, r_decay
        )
        expression = propagator * production * decay_factor
    dynamics_symbol = sp.Symbol(
        Rf"\mathcal{{R}}^{{{chain.resonance.latex}}}_{{{orbital}}}"
    )
    return DefinedExpression(dynamics_symbol, parameters, {dynamics_symbol: expression})


def unnormalized_form_factor(s, m1, m2, orbital, radius):
    normalization = sp.Abs(SphericalHankel1(sp.Integer(orbital), sp.S.One).doit())
    return FormFactor(s, m1, m2, orbital, radius) / normalization

In [ ]:
example_chain = next(chain for chain in chains if chain.resonance.name == "L1520")
Math(aslatex(formulate_dynamics(example_chain).subexpressions))

## Model formulation

Select LS couplings at both vertices and use one complex coefficient per LS chain. The coefficient symbol carries the resonance name, followed by the production $(L,S)$ and decay $(l,1/2)$ indices. Assigning the weights after formulation keeps the parameters available for later variation.

Subsystem 3 is the natural alignment frame because every chain has a pK isobar. The general builder sums over all spin-one projections. A real photon has only $\lambda_\gamma=\pm1$, so we explicitly restrict the outer intensity sum to these two helicities. No rotation of the photon to a different subsystem is required in this model.

The reference convention includes an explicit $\sqrt{2J_R+1}$ multiplying each chain amplitude. The general builder uses the same normalized LS recoupling factors but leaves this factor in the coefficient. We therefore assign $c_\mathrm{builder}=\sqrt{2J_R+1}\,c_\mathrm{reference}$.

In [ ]:
builder = DalitzPlotDecompositionBuilder(decay, min_ls=False)
for resonance in resonances.values():
    builder.dynamics_choices.register_builder(resonance.name, formulate_dynamics)
model = builder.formulate(reference_subsystem=3, use_coefficients=True)
assigned_coefficients = set()
for (name, orbital, spin), weight in ls_weights.items():
    resonance = resonances[name]
    decay_orbital = resonance_parameters[name][4]
    coefficient = sp.IndexedBase(Rf"\mathcal{{H}}^\mathrm{{LS,{resonance.latex}}}")[
        orbital, sp.Rational(spin), decay_orbital, sp.Rational(1, 2)
    ]
    assert coefficient in model.parameter_defaults
    model.parameter_defaults[coefficient] = (
        np.sqrt(float(2 * resonance.spin + 1)) * weight
    )
    assigned_coefficients.add(coefficient)
assert len(assigned_coefficients) == len(chains)
photon_helicity = sp.Symbol("lambda3", rational=True)
model = evolve(
    model,
    intensity=PoolSum(
        model.intensity.expression,
        *(
            (symbol, (-1, 1) if symbol == photon_helicity else values)
            for symbol, values in model.intensity.indices
        ),
    ),
)
model.intensity

The model now contains the reference LS weights and dynamics, with a transverse photon in the intensity sum. This is an explicit DPD reconstruction. Comparisons of absolute intensities with another implementation also require matching helicity sums, alignment conventions, and normalization; the Dalitz plot below is normalized independently.

## Numerical evaluation

Substitute the numerical defaults and transform the invariant masses to the helicity angles required by the amplitudes. The third invariant follows from $\sigma_1+\sigma_2+\sigma_3=m_{\Lambda_b}^2+m_p^2+m_K^2$.

In [ ]:
sigma1, sigma2, sigma3 = sp.symbols("sigma1:4", nonnegative=True)
definitions = dict(model.variables)
definitions[sigma2] = model.invariants[sigma2]
definitions = {
    symbol: expression.xreplace(definitions).xreplace(model.parameter_defaults)
    for symbol, expression in definitions.items()
}
transformer = SympyDataTransformer.from_sympy(definitions, backend="numpy")
intensity_expression = cached.xreplace(cached.unfold(model), model.parameter_defaults)
intensity_function = cached.lambdify(intensity_expression, backend="numpy")

## Dalitz plot

The axes are $\sigma_1=m^2(K^-\gamma)$ and $\sigma_3=m^2(pK^-)$. We restrict $m(pK^-)$ to $2.5\,\mathrm{GeV}$, the upper limit of the published analysis {cite}`LHCb:2024vtc`, and evaluate only the physical interior. This resolves the narrow $\Lambda(1520)$ band while keeping the model within the studied mass range. The resonance bands illustrate the pK dynamics and their coherent interference; this plot includes neither detector efficiency nor event selection.

In [ ]:
parent_mass = decay.initial_state.mass
m1, m2, m3 = (decay.final_state[i].mass for i in (1, 2, 3))
x = np.linspace((m2 + m3) ** 2, (parent_mass - m1) ** 2, 401)
y = np.linspace((m1 + m2) ** 2, 2.5**2, 401)
X, Y = np.meshgrid(x, y)
pair_mass = np.sqrt(Y)
energy2 = (Y + m2**2 - m1**2) / (2 * pair_mass)
energy3 = (parent_mass**2 - Y - m3**2) / (2 * pair_mass)
momentum2 = np.sqrt(np.maximum(energy2**2 - m2**2, 0))
momentum3 = np.sqrt(np.maximum(energy3**2 - m3**2, 0))
center = m2**2 + m3**2 + 2 * energy2 * energy3
half_width = 2 * momentum2 * momentum3
physical = (center - half_width < X) & (center + half_width > X)
data = {"sigma1": X[physical], "sigma3": Y[physical]}
data.update(transformer(data))
intensities = np.full(X.shape, np.nan)
intensities[physical] = intensity_function(data)
assert np.all(np.isfinite(intensities[physical]))
assert np.all(intensities[physical] >= 0)
assert np.nanmax(intensities) > 0

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6), constrained_layout=True)
mesh = ax.pcolormesh(X, Y, intensities / np.nansum(intensities), rasterized=True)
ax.set_xlabel(R"$\sigma_1 = m^2(K^-\gamma)$ [GeV$^2$]")
ax.set_ylabel(R"$\sigma_3 = m^2(pK^-)$ [GeV$^2$]")
fig.colorbar(mesh, ax=ax, label="Normalized intensity (a.u.)")
plt.show()